## Smart Grid Regression - Notebook

###  Carrega dados (SIN_dataset.xlsx se existir), faz EDA, treina pipelines (Ridge + RF) e salva modelo. Projetado para dados de barras/linhas (P_inj, Q_inj, V, theta, R_line, X_line, etc).

## Autor: Pedro Victor Veras (template entregue pelo assistente)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

plt.style.use('seaborn-darkgrid')

In [ ]:
# ## 1) Carregar dados
# - Se `data/SIN_dataset.xlsx` existir, ele será carregado.
# - Caso contrário, geramos um dataset sintético de exemplo.

DATA_PATH = os.path.join('data', 'SIN_dataset.xlsx')

if os.path.exists(DATA_PATH):
    print(f"Carregando dataset real: {DATA_PATH}")
    df = pd.read_excel(DATA_PATH)
else:
    print("Arquivo não encontrado. Gerando dataset sintético de exemplo (16 barras).")
    rng = np.random.RandomState(42)
    N = 1000  # número de amostras (cenários)
    # Colunas de exemplo: P_inj_barra_i, Q_inj_barra_i, V_barra_i, theta_barra_i, R_line, X_line, carga_total_regiao
    # Para simplificar, vamos gerar features agregadas
    df = pd.DataFrame({
        'P_total': rng.uniform(1000,
    20000, size=N),          # MW
        'Q_total': rng.uniform(-500,
    2000, size=N),          # MVAr
        'V_min': rng.uniform(0.95,
    1.05, size=N),
        'V_max': rng.uniform(0.95,
    1.07, size=N),
        'I_max': rng.uniform(0.0,
    600.0, size=N),
        'n_gens': rng.integers(1,
    50, size=N),
        'n_loads': rng.integers(10,
    500, size=N),
        'losses': rng.uniform(1.0,
    500.0, size=N),
})
    # Target: um atributo útil — por exemplo,
#"tensão média crítica" ou "violação boolean convertida em escala"
    # Vamos sintetizar um target que depende de P_total, Q_total e perdas

df['V_critical'
] = 1.0 - 0.00001*(df['P_total'
] - 5000) - 0.00002*df['losses'
] + rng.normal(0,
0.005, size=N)
# se quiser, exporte pra data/SIN_dataset.xlsx para testar com seu pipeline
os.makedirs('data', exist_ok=True)
df.to_excel(DATA_PATH, index=False)
print(f"Dataset sintético salvo em {DATA_PATH}")

df.head()

print("Shape:", df.shape)
display(df.describe().T)

# pares e correlações
plt.figure(figsize=(10,
8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Matriz de correlação")
plt.tight_layout()
plt.show()


SyntaxError: invalid syntax (4292122272.py, line 35)

## Treinamento ML

In [ ]:
TARGET_COLUMN = 'V_critical'
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Coluna target '{TARGET_COLUMN}' não encontrada no dataset.")

X = df.drop(columns=[TARGET_COLUMN
])
y = df[TARGET_COLUMN
]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train/Test shapes:", X_train.shape, X_test.shape)


pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('ridge', Ridge())
])

params_ridge = {
    'poly__degree': [
        1,
        2
    ],            
    # 1 = linear,

}

gs_ridge = GridSearchCV(pipe_ridge, params_ridge, cv=4, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
gs_ridge.fit(X_train, y_train)
print("Melhor ridge:", gs_ridge.best_params_,
"score:", gs_ridge.best_score_)

# Avaliação
def eval_model(model, X_test, y_test, name='model'):
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"{name} - RMSE: {rmse:.5f}, MAE: {mae:.5f}, R2: {r2:.5f}")
    return y_pred

y_pred_ridge = eval_model(gs_ridge.best_estimator_, X_test, y_test, 'Ridge (best)')

plt.figure(figsize=(6,
6))
plt.scatter(y_test, y_pred_ridge, s=10, alpha=0.6)
plt.plot([y_test.min(), y_test.max()
],
[y_test.min(), y_test.max()
], 'r--')
plt.xlabel('Real')
plt.ylabel('Predito')
plt.title('Ridge: Real x Predito')
plt.show()



In [ ]:

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),  # RF não precisa, mas mantemos para consistência
    ('rf', RandomForestRegressor(random_state=42))
])

params_rf = {
    'rf__n_estimators': [
        100,
        200
    ],
    'rf__max_depth': [
        6,
        12, None
    ],
    'rf__min_samples_leaf': [
        1,
        4
    ]
}

gs_rf = GridSearchCV(pipe_rf, params_rf, cv=4, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
gs_rf.fit(X_train, y_train)
print("Melhor RF:", gs_rf.best_params_,
"score:", gs_rf.best_score_)

# %% [code
]
y_pred_rf = eval_model(gs_rf.best_estimator_, X_test, y_test, 'RandomForest (best)')

plt.figure(figsize=(6,
6))
plt.scatter(y_test, y_pred_rf, s=10, alpha=0.6)
plt.plot([y_test.min(), y_test.max()
],
[y_test.min(), y_test.max()
], 'r--')
plt.xlabel('Real')
plt.ylabel('Predito')
plt.title('RandomForest: Real x Predito')
plt.show()


In [ ]:
rf_model = gs_rf.best_estimator_.named_steps['rf'
]
feature_names = X_train.columns
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(8,
5))
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title("Importância das features (RandomForest)")
plt.tight_layout()
plt.show()

In [ ]:

os.makedirs('models', exist_ok=True)
joblib.dump(gs_rf.best_estimator_, 'models/best_model_rf.joblib')
joblib.dump(gs_ridge.best_estimator_, 'models/best_model_ridge.joblib')
print("Modelos salvos em ./models/")

# salvar previsões e comparação
results = X_test.copy()
results['y_true'
] = y_test.values
results['y_pred_rf'
] = y_pred_rf
results['y_pred_ridge'
] = y_pred_ridge
results.to_csv('models/predictions.csv', index=False)
print("Predições salvas em models/predictions.csv")


# 8) Próximos passos sugeridos
# - Substituir o dataset sintético pelo `SIN_dataset.xlsx` real (coloque em data/ e ajuste TARGET_COLUMN)
# - Adicionar engenharia de features específicas do Pandapower (ex.: P_inj_por_barra, reatividade, impedâncias por linha)
# - Transformar o notebook em script modular (`src/`) e um endpoint FastAPI para servir predições
# - Visualizar resultados num dashboard (Streamlit / Dash) ou gerar PDF com matplotlib